# **Assignment 2: Employee Attrition Risk Analyzer**
##### A Decision-Tree Predictor with a Review Carousel

**CMPUT 175 - Winter 2026**
##### **Created By**: `Maryna Shumeiko`

You are a human resources (HR) analyst at Yegstar Forge Systems, and you’ve been asked to build a retention plan that is targeted, defensible, and fast to execute. The company has limited retention resources, so HR must decide who to prioritize for interventions such as workload adjustments, flexible schedules, compensation reviews, or a promotion conversation. To do this properly, you need more than opinions; you need evidence. You are given HR data materialized in a set of text files, based on a fictional employee-attrition dataset created by IBM data scientists. 

##### **This program:** 
- analyze data quality and possible bias
- train a simple model that predicts who is most likely to quit
- deploy a review workflow that lets HR quickly cycle through employees, flag high-risk cases, and decide what retention action to take before competitors recruit them away.

In [ ]:
NUM_TYPE = 'numeric'
CATEG_TYPE = 'categorical'
ORDINAL_TYPE = 'ordinal'
NUM_ORDINAL = [NUM_TYPE, ORDINAL_TYPE]

COL_NAMES_TYPES = { 'Age': NUM_TYPE,
                    'Attrition': CATEG_TYPE,
                    'BusinessTravel': CATEG_TYPE,
                    'Department': CATEG_TYPE,
                    'DistanceFromHome': NUM_TYPE,
                    'EnvironmentSatisfaction': ORDINAL_TYPE,
                    'Gender': CATEG_TYPE,
                    'JobInvolvement': ORDINAL_TYPE,
                    'JobRole': CATEG_TYPE,
                    'JobSatisfaction': ORDINAL_TYPE,
                    'NumCompaniesWorked': NUM_TYPE,
                    'MonthlyIncome': NUM_TYPE,
                    'OverTime': CATEG_TYPE,
                    'PercentSalaryHike': NUM_TYPE,
                    'WorkLifeBalance': ORDINAL_TYPE,
                    'YearsAtCompany': NUM_TYPE,
                    'YearsInCurrentRole': NUM_TYPE,
                    'YearsSinceLastPromotion': NUM_TYPE,
                    'EmployeeName': CATEG_TYPE}

# **Section A — Training Data Pre-Processing and Analysis**

## 1. Identifying and Handling Missing Values

In [ ]:
def read_file(filename):
    '''
    Parameter: filename: the name of the text file to read, type str
    - reads the data from the file specified by filename
    - takes the 1st line as column names
    - splits each line at commas and processes its items:
        - converts to float if numeric type, to int if ordinal type
          and appends the value to the corresponding column list
          if conversion fails, appends None
        - otherwise appends item as a string
    Returns a tuple with data dictionary, and the column names list
    '''
    with open(filename, 'r') as file:
        column_names = file.readline().strip().split(',')
        data = {key: [] for key in column_names}
        for line in file:
            items = line.strip().split(',')
            for i in range(len(items)):
                if COL_NAMES_TYPES[column_names[i]] == NUM_TYPE:
                    try: 
                        data[column_names[i]].append(float(items[i]))
                    except ValueError:
                        data[column_names[i]].append(None)
                elif COL_NAMES_TYPES[column_names[i]] == ORDINAL_TYPE:
                    try: 
                        data[column_names[i]].append(int(float(items[i])))
                    except ValueError:
                        data[column_names[i]].append(None)
                else: data[column_names[i]].append(items[i])
    return data, column_names

Count the number of rows (excluding the header)

In [ ]:
train_data, train_col_names = read_file('train.csv')
initial_rows_num = len(train_data[train_col_names[0]])

Identify the columns that have missing values in a categorical column, and how many are missing in each.

In [ ]:
categ_missing = {}
for column_name in train_data:
    if COL_NAMES_TYPES[column_name] == CATEG_TYPE:
        missing_count = train_data[column_name].count('')
        if missing_count != 0:
            categ_missing[column_name] = missing_count

Remove the entire row where a categorical column contains a missing value.

In [ ]:
rows_removed_count = 0
for column_name in train_data:
    if COL_NAMES_TYPES[column_name] == CATEG_TYPE:
        missing_indices = []
        for i in range(len(train_data[column_name])):
            if train_data[column_name][i] == '':
                missing_indices.append(i)
        for row_num in reversed(missing_indices):
            for value in train_data.values():
                del value[row_num]
            rows_removed_count += 1

Identify the columns that have missing values in a numeric/ordinal column, and how many are missing in each. Impute the missing value by replacing it with the average of that numeric/ordinal column, computed from the non-missing values in the training data.

In [ ]:
num_ordinal_missing = {}
for column_name in train_data:
    if COL_NAMES_TYPES[column_name] in NUM_ORDINAL:
        missing_count = 0
        sum_of_val = 0
        num_of_val = 0

        for value in train_data[column_name]:
            if value is not None:
                sum_of_val += value
                num_of_val += 1
        
        mean = round(sum_of_val / num_of_val, 2) 
        
        if COL_NAMES_TYPES[column_name] == ORDINAL_TYPE:
            mean = int(mean)
        
        for i in range(len(train_data[column_name])):
            if train_data[column_name][i] == None:
                missing_count += 1
                train_data[column_name][i] = mean
        
        if missing_count != 0:
            num_ordinal_missing[column_name] = missing_count

Confirmation that there are no missing values left in the columns you are using.

In [ ]:
no_missing = True
for column_name in train_data:
    if None in train_data[column_name] or '' in train_data[column_name]:
        no_missing = False

Print the results

In [ ]:
print(f'Initial number of rows: {initial_rows_num}')

for col_name, num_missing in num_ordinal_missing.items(): 
    print(f'{col_name}: {num_missing} values missing (imputed)')

for categ_col, num_missing in categ_missing.items(): 
    print(f'{categ_col}: {num_missing} values missing (rows removed)')

print(f'Rows removed due to categorical missing values: {rows_removed_count}')

print(f'Remaining number of rows: {len(train_data[train_col_names[0]])}')

if no_missing:
    print('\nConfirmed: No missing values remain in the dataset.')

## 2. Analyzing Attrition Distribution by Age (Bar Chart)

### Analyze attrition distribution by age in the training data. 
- Group employees into 10-year age buckets (<20, 20–29, 30–39, 40-49, 50-59, 60+)
- Separate them into currently “Attrition = Yes” and “Attrition = No” categories using the Attrition attribute
- Using matplotlib, plot two bar charts showing the age distribution for each category:
    - x-axis: Age Buckets
    - y-axis: Number of Employees

In [ ]:
import matplotlib.pyplot as plt

categories = ['<20', '20-29', '30-39', '40-49', '50-59', '60+']
step = 10
attrition_yes = [0] * len(categories)
attrition_no = [0] * len(categories)

col_name_age = 'Age'
col_name_attrition = 'Attrition'
attrition_val_1 = 'Yes'
attrition_val_2 = 'No'
for i in range(len(train_data[col_name_age])):
    age_value = train_data[col_name_age][i]
    if age_value < 20:
        index = categories.index('<20')
    elif age_value >= 60:
        index = categories.index('60+')
    else: index = int((age_value//step))-1 
    if train_data[col_name_attrition][i] == attrition_val_1:
        attrition_yes[index] += 1
    else: attrition_no[index] += 1 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Plot Attrition = Yes
rects1 = ax1.bar(categories, attrition_yes, color='red')
ax1.set_title('Attrition = Yes')
ax1.set_ylabel('Number of Employees')
ax1.set_xlabel('Age Bucket')
ax1.grid(True, axis='y')

# Plot Attrition = No
rects2 = ax2.bar(categories, attrition_no, color='green')
ax2.set_title('Attrition = No')
ax2.set_ylabel('Number of Employees')
ax2.set_xlabel('Age Bucket')
ax2.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 3. Analyzing Attrition Among Employees based on Job Satisfaction (Pie Chart)

### Analyze whether employees with low job satisfaction quit more often than employees with high job satisfaction in the training data.
- First, split employees into two subgroups using the JobSatisfaction column:
    - Low satisfaction group: JobSatisfaction is 1 or 2
    - High satisfaction group: JobSatisfaction is 3 or 4
- For each subgroup, count how many employees have Attrition = Yes and how many have Attrition = No. 
- Using matplotlib, create two pie charts:
    - A pie chart for the low satisfaction group showing the percentage who quit versus the percentage who stayed.
    - A pie chart for the high satisfaction group showing the percentage who quit versus the percentage who stayed.

In [ ]:
col_name_satisfaction = 'JobSatisfaction'

low = [0,0]   # [attrition = yes, attrition = no] for Low Job Satisfaction (1-2)
high = [0,0]  # [attrition = yes, attrition = no] for High Job Satisfaction (3-4)

for i in range(len(train_data[col_name_satisfaction])):
    if train_data[col_name_attrition][i] == attrition_val_1:
        index = 0
    else: index = 1
    if train_data[col_name_satisfaction][i] in [1,2]:
        low[index] += 1
    else: high[index] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
labels = ['Attrition = Yes', 'Attrition = No']

# Plot first pie chart Low Job Satisfaction (1-2)
ax1.pie(low, labels=labels, autopct='%1.1f%%', startangle=140, colors=['#ffc0cb','#00ffff'])
ax1.set_title('Low Job Satisfaction (1-2)')

# Plot second pie chart High Job Satisfaction (3-4)
ax2.pie(high, labels=labels, autopct='%1.1f%%', startangle=140, colors=["#ffc0cb","#00ffff"])
ax2.set_title('High Job Satisfaction (3-4)')

plt.tight_layout()
plt.show()

## 4. Checking for Class Imbalance in the Training Data 

Determine whether the training data is balanced. 
We want to predict Attrition. So we must consider if we have the same number of employees who quit vs stayed in our training data.
Count them and display the counts. If one class (quitting) is much less common than the other, determine the potential impact on your model’s performance. 

In [ ]:
print(f'Number of employees who quit (Attrition=Yes): {low[0] + high[0]}')
print(f'Number of employees who stayed (Attrition=No): {low[1] + high[1]}')

# **Section B — Building the Decision Tree Classifier**

## 1. Preparing the Data for Training

Fit the scaler on the training features (all numerical columns for this assignment).
Use the same scaler to transform both the training and testing features (all numerical columns for this assignment) before training your Decision Tree.

In [ ]:
def get_unscaled_num_features(data, col_names):
    '''
    Parameters:
        - data: dictionary with column names as keys, and
                values are lists of all values for the specified by key column name
        - col_names: a list of column names in the data
    - Iterates through each row of the data to extract numerical and ordinal columns (features)
    Returns a list of lists containing the unscaled numerical and ordinal features
    '''
    unscaled_features = []
    for i in range(len(data[col_names[0]])):
        feature_row = []
        for col_name, value in data.items():
            if COL_NAMES_TYPES[col_name] in NUM_ORDINAL:
                feature_row.append(value[i])
        unscaled_features.append(feature_row)
    return unscaled_features

Preparing the train file

In [ ]:
from sklearn.preprocessing import StandardScaler

unscaled_train_features = get_unscaled_num_features(train_data, train_col_names)

# Create and apply StandardScaler to scale the data
scaler = StandardScaler()
scaled_train_features = scaler.fit_transform(unscaled_train_features)


Preparing the test file 

In [ ]:
test_data, test_col_names = read_file('test.csv')

unscaled_test_features = get_unscaled_num_features(test_data, test_col_names)

scaled_test_features = scaler.transform(unscaled_test_features)

## 2. Building the Decision Tree Classifier

### 2.1. Training the Model

To train the Decision Tree model, create two lists: x_train and y_train. 

- The x_train list will contain the input features (all numerical columns for this assignment) used to train the model, where each item is a sublist of the scaled HR feature values for one employee. 

- The y_train list will contain the Attrition label for each employee in x_train, encoded as:
    - 1 if Attrition = Yes (the employee quit)
    - 0 if Attrition = No (the employee stayed)

Next, use DecisionTreeClassifier from scikit-learn to train the model by calling the fit() method.

In [ ]:
def code_attition(data):
    '''
    Parameter:
        - data: dictionary with column names as keys, and
                values are lists of all values for the specified by key column name
    - iterates through the attrition column in the data
    - encoded as:
        - 1 if Attrition = Yes (the employee quit)
        - 0 if Attrition = No (the employee stayed)
    Returns a list of binary integers representing the attrition status
    '''
    codes = []
    for value in data[col_name_attrition]:
        if value == attrition_val_1:
            codes.append(1)
        else: codes.append(0)
    return codes

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(random_state=42)  # Create the model

x_train = scaled_train_features

y_train = code_attition(train_data)

clf.fit(x_train, y_train)  # Train the model with the dat

### 2.2. Evaluating the Model

#### Testing the model on a separate test set (use the file test.csv). 

- Create two lists: x_test and y_test. 
    - x_test list contains the same feature columns you used for training (in the same order)
    - y_test contains the true attrition outcomes from the file, encoded as:
        - 1 if Attrition = yes
        - 0 if Attrition = no
                           
- Once test data x_test is prepared, the classifier can make predictions on x_test using the predict() method. This produces y_pred, which contains the predicted attrition label (1 predicted to quit, 0 predicted to stay) for each employee in the test set.

##### To evaluate the model, compute:

- Accuracy: the percentage of employees whose attrition outcome was predicted correctly. Round to two decimal places.

- Precision, Recall, and F1-score:  
    - Precision: Of all employees predicted to quit, how many actually quit?
    - Recall: Of all employees who actually quit, how many did the model correctly predict?
    - F1-score: A balance between precision and recall (useful when both are important).

- Confusion Matrix: A 2×2 table that shows the kinds of mistakes the model makes.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

x_test = scaled_test_features
y_test = code_attition(test_data)

# Get predictions from the trained model
y_pred = clf.predict(x_test)


# Print evaluation metrics

print("Test Accuracy:", round(accuracy_score(y_test, y_pred),2))

print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# **Section C —Deployment of the Predictor**

### 1. Prediction

- Extract the required feature columns from NewEmployees.csv. (This file has the same columns and or1der as the training/testing files, but it includes one additional column at the beginning called EmployeeName, and it does not include any extra target labels beyond what is already in the provided format.)

- Read NewEmployees.csv and build a list of employee records. For each record, keep the employee’s name from EmployeeName, and extract the same feature columns you used in Section B (in the same order). Scale these feature values using the same scaler you fit on the training data. Then, use your trained decision tree model to make predictions for every employee in NewEmployees.csv.

- For each employee record, your model must output a simple prediction:
    - “Yes” = likely to quit
    - “No” = likely to stay
- Store the prediction alongside the employee’s data so it can be displayed later in the carousel. - Finally, print a confirmation showing how many employees were processed and how many predictions were generated.

In [ ]:
new_employees_data, new_employees_col_names = read_file('NewEmployees.csv')

unscaled_new_employees_features = get_unscaled_num_features(new_employees_data, new_employees_col_names)

scaled_new_employees_features = scaler.transform(unscaled_new_employees_features)

# Get predictions from the trained model
y_pred_new_employees = clf.predict(scaled_new_employees_features)

yes_no_pred = []
for value in y_pred_new_employees:
    if value == 1:
        yes_no_pred.append('Yes')
    else: yes_no_pred.append('No')
new_employees_data['Prediction'] = yes_no_pred  

# printing confirmation messages
print(f'Processed {len(new_employees_data[new_employees_col_names[0]])} employees')
print(f'Generated {len(y_pred_new_employees)} predictions')

### 3. Interface

#### Build a simple text-based interface to navigate through a list of employee records using the methods in the Carousel class. 
Each employee will be displayed one at a time, and the user will be able to move through the carousel using commands to go to the next or previous employee. The interface will simulate a vertical carousel by showing one record at a time, along with the employee’s details and the model’s predicted attrition (“Yes” or “No”).


Before printing the carousel, prompt the user to press Enter, ensuring all outputs are displayed before the carousel appears. Users should be able to interact with the carousel by entering “1” to go next, “2” to go back, “3” to flag the current employee for follow-up, “4” to switch between the active carousel and the flagged view, and “0” to quit. If the user enters an invalid input, keep prompting them until a valid selection is made.


Your carousel should clear the previous screen when going to the next screen. Remember Lab 2, where we used clear() to clean the terminal. We will use a similar function here. Import clear_output from IPython.display and call clear_output(wait=True) at the start of each loop iteration (before printing the next screen).

In [ ]:
def print_employee_info (name):
    '''
    Parameter:
        - name: the string name of the employee to display
    - identifies the row index corresponding to the employee in the data
    - iterates through the data and prints all the attributes except the employee name
    - determines recommendation message based on the prediction result
    - prints formatted summary and navigation instructions
    '''

    separator = '-' * 50
    print(separator,f'\nEmployee Name: {name}\n',separator)

    row_index = new_employees_data[new_employees_col_names[0]].index(name)
    
    for col_name, value in new_employees_data.items():
        if col_name not in ['EmployeeName','Prediction']:
            print(f'  {col_name}: {value[row_index]}')
    
    if new_employees_data['Prediction'][row_index] == 'Yes':
        recom = 'Flag for Retention'
    else: recom = 'No Action Needed'

    print(separator)
    print(f'Prediction (Likely to Quit): {new_employees_data['Prediction'][row_index]}')
    print(f'Recommend: {recom}')
    print(separator)

    print('\nEnter 1 for Next; 2 for Previous; 3 to Flag; 4 to Switch View; 0 to Quit')

In [ ]:
def print_flagged(flagged):
    '''
    Parameter: 
        - flagged: a list of employees names flagged for follow-up
    - iterates through the flagged list to get data for each flagged employee
    - identifies row_index in the data for each name to get prediction
    - prints a numbered list of employees including their names and attrition predictions
    - prints navigation instructions
    '''
    separator = '-' * 50
    print(separator,f'\nFLAGGED EMPLOYEES FOR RETENTION PLANNING\n',separator)

    for name in flagged:
        row_index = new_employees_data[new_employees_col_names[0]].index(name)
        print(f'{flagged.index(name) + 1}. {name} - Prediction: {new_employees_data['Prediction'][row_index]}')
    
    print(separator)
    print('\nEnter 1 for Next; 2 for Previous; 3 to Flag; 4 to Switch View; 0 to Quit')

In [ ]:
from IPython.display import clear_output
from carousel import Carousel

carousel = Carousel()
for employee in new_employees_data[new_employees_col_names[0]]:
    carousel.add(employee)
carousel.moveNext()

input('Press Enter')

quit = False

flagged = []
employee_screen = True

while not quit:
    clear_output(wait=True)

    if employee_screen:
        print_employee_info(carousel.getCurrentData())
    else: print_flagged(flagged)

    not_valid = True
    while not_valid:
        choice = input('Enter choice: ')
        if choice not in ['0','1','2','3','4']:
            print('Invalid Input.')
        else: not_valid = False
    
    if choice == '1' and employee_screen:
        carousel.moveNext()     
    
    elif choice == '2' and employee_screen:
        carousel.movePrevious()
    
    elif choice == '3' and employee_screen:
        if carousel.getCurrentData() not in flagged:
            flagged.append(carousel.getCurrentData())
    
    elif choice == '4':
        employee_screen = not employee_screen
    
    if choice == '0':
        quit = True
